This code is for sampling a given point shapefile in GEE to append valid L7, L8, L30, S30, and Dynamic world values. Pixel values are considered valid if the are not affected by clouds, or not no data. Dates where there are valid pixels for all datasets (+/-1 day for L7) are randomly queried until an instance that is valid for all datasets is selected and sampled. 

In [1]:
import geopandas as gpd,ee,re,time,ast
from datetime import timedelta, datetime as _dt
import os,time,random,joblib,numpy as np,pandas as pd
from joblib import Parallel,delayed
from contextlib import contextmanager
from itertools import product
try: from tqdm.auto import tqdm
except: tqdm=None
import ee
from pathlib import Path

ee_project="ee-gamr05055"
L7COL='LANDSAT/LE07/C02/T1_L2'
L8COL="LANDSAT/LC08/C02/T1_L2"
S30COL="NASA/HLS/HLSS30/v002"
L30COL="NASA/HLS/HLSL30/v002"
DWCOL="GOOGLE/DYNAMICWORLD/V1"
Landsat_bands=['.*B[0-9].*','QA_PIXEL','QA_RADSAT']
S30_Bands=['B1','B2','B3','B4','B5','B6','B7','B8','B8A','B9','B10','B11','B12','Fmask']
L30_Bands=['B1','B2','B3','B4','B5','B6','B7','Fmask']
DW_band=['label']
ALLOWED_QA_PIXEL={5440,21824,5504,21952,13600,30048}
ee.Authenticate()
ee.Initialize(project=ee_project)

In [ ]:
#Input reference data
L7_dates=pd.read_csv(r'E:\GIS\Landcover_sampling\Reference\L7_dates50.csv')
L8_dates=pd.read_csv(r'E:\GIS\Landcover_sampling\Reference\L8_dates50.csv')
S30_dates=pd.read_csv(r'E:\GIS\Landcover_sampling\Reference\S30_dates50.csv')
WRS_centroids=r"E:\GIS\Landsat Normalization\Landsat_WRS_index\WRS2_descending_centroid.shp"
MGRS_centroids=r'E:\GIS\Landsat Normalization\Landsat_WRS_index\sentinel_2_index_centroid.shp'
#sometime you need to re-download WRS2_descending.shp from USGS.
WRS_shp=r"E:\GIS\Landsat Normalization\Landsat_WRS_index\WRS2_descending.shp"
Sen2_shp= r"E:\GIS\Landsat Normalization\Landsat_WRS_index\sentinel_2_index_shapefile.shp",

In [1]:
# Points are alreadly located at L7L8 intersections
def list_tile_intersections(points,wrs,mgrs,path_col='PATH',row_col='ROW',mgrs_col=None,keep_point_cols=True,n_jobs=-1):
    _g=lambda x:gpd.read_file(x) if isinstance(x,(str,os.PathLike)) else x
    pts=_g(points).copy(); wrs=_g(wrs).copy(); mgrs=_g(mgrs).copy()
    if mgrs_col is None:
        for c in('MGRS','mgrs','NAME','Name','name','TILE','Tile','ID','grid','GRID','MGRS_TILE','Name'):
            if c in mgrs.columns: mgrs_col=c; break
        else: raise KeyError('mgrs_col not found')
    wrs=wrs[[path_col,row_col,'geometry']].to_crs(pts.crs); mgrs=mgrs[[mgrs_col,'geometry']].to_crs(pts.crs)
    siw,sim=wrs.sindex,mgrs.sindex
    uniq=lambda a:sorted(pd.Series(a).dropna().unique().tolist())
    def f(i,p):
        cw=list(siw.query(p)) if hasattr(siw,'query') else list(siw.intersection(p.bounds))
        cm=list(sim.query(p)) if hasattr(sim,'query') else list(sim.intersection(p.bounds))
        w=wrs.iloc[cw]; m=mgrs.iloc[cm]
        if len(w): w=w.loc[w.geometry.intersects(p)]
        if len(m): m=m.loc[m.geometry.intersects(p)]
        return i, (uniq(w[path_col]) if len(w) else []), (uniq(w[row_col]) if len(w) else []), (uniq(m[mgrs_col]) if len(m) else [])
    res=Parallel(n_jobs=n_jobs,prefer='threads',batch_size=256)(delayed(f)(i,p) for i,p in zip(pts.index,pts.geometry))
    out=(pts if keep_point_cols else pts[['geometry']]).copy(); n=len(out)
    out['paths']=[[] for _ in range(n)]; out['rows']=[[] for _ in range(n)]; out['mgrs']=[[] for _ in range(n)]
    for i,pa,ro,mg in res: out.at[i,'paths']=pa; out.at[i,'rows']=ro; out.at[i,'mgrs']=mg
    return gpd.GeoDataFrame(out,geometry='geometry',crs=pts.crs)
def attach_point_dates(point_gdf,L7_dates,L8_dates,S30_dates,
                       path_col="paths",row_col="rows",mgrs_col="mgrs",
                       l7_cols=("WRS_PATH","WRS_ROW","dates_by_pr"),
                       l8_cols=("WRS_PATH","WRS_ROW","dates_by_pr"),
                       s30_cols=("mgrs","dates_by_mgrs"),njobs=-1):
    _as_list=lambda v:([] if v is None or (isinstance(v,float) and pd.isna(v)) else (list(v) if isinstance(v,(list,tuple,set)) else v.tolist() if isinstance(v,np.ndarray) else (ast.literal_eval(v) if isinstance(v,str) and v.strip().startswith('[') else ([t.strip(" '\"\t") for t in v.strip("[]").split(',')] if isinstance(v,str) and ',' in v else ([v] if isinstance(v,str) and v.strip() else [])))))
    def _norm_dates(seq):
        if not seq: return ()
        s=pd.to_datetime(pd.Series(seq),errors="coerce").dropna().dt.strftime("%Y-%m-%d")
        return tuple(sorted(set(s.tolist())))
    l7_lut={(int(p),int(r)):_norm_dates(_as_list(d)) for p,r,d in L7_dates[list(l7_cols)].itertuples(index=False,name=None)}
    l8_lut={(int(p),int(r)):_norm_dates(_as_list(d)) for p,r,d in L8_dates[list(l8_cols)].itertuples(index=False,name=None)}
    s30_lut={str(t).strip().upper().lstrip("T"):_norm_dates(_as_list(d)) for t,d in S30_dates[list(s30_cols)].itertuples(index=False,name=None)}
    df=point_gdf
    P=[[int(p) for p in _as_list(v) if pd.notna(p)] for v in (df[path_col] if path_col in df.columns else [None]*len(df))]
    R=[[int(r) for r in _as_list(v) if pd.notna(r)] for v in (df[row_col]  if row_col  in df.columns else [None]*len(df))]
    T=[[str(t).strip().upper().lstrip("T") for t in _as_list(v) if pd.notna(t) and str(t).strip()] for v in (df[mgrs_col] if mgrs_col in df.columns else [None]*len(df))]
    _merge=lambda seqs: (sorted(set(d for s in seqs if s for d in s)) if seqs else [])
    def _row(p,r,t):
        PR=set(product(p,r)) if p and r else ()
        return {"L7_d":_merge([l7_lut.get(k,()) for k in PR]),
                "L8_d":_merge([l8_lut.get(k,()) for k in PR]),
                "S30_d":_merge([s30_lut.get(u,()) for u in t])}
    rows=Parallel(n_jobs=njobs,prefer="threads")(delayed(_row)(p,q,t) for p,q,t in zip(P,R,T))
    return df.join(pd.DataFrame(rows,index=df.index))
def attach_good_dates_exact(gdf,l7_col="L7_d",l8_col="L8_d",s30_col="S30_d",
                            out_col="good_d",tol_days=1,stats_csv=None,njobs=-1,remove_empty=True):
    def _to_days(lst):
        if not lst: return np.array([],dtype="int64")
        return pd.to_datetime(list(lst),errors="coerce").dropna().values.astype("datetime64[D]").astype("int64")
    def _row(L7,L8,S30):
        A=_to_days(L8); B=_to_days(S30); C=np.sort(_to_days(L7))
        if A.size==0 or B.size==0 or C.size==0: return []
        base=np.intersect1d(A,B); out=set()
        for d in base:
            lo=np.searchsorted(C,d-tol_days,"left"); hi=np.searchsorted(C,d+tol_days,"right")
            if hi>lo: out.add(int(d)); out.update(map(int,C[lo:hi]))
        return [pd.Timestamp(np.datetime64(x,"D")).strftime("%Y-%m-%d") for x in sorted(out)]
    def _counts_row(s):
        A=np.sort(_to_days(s.get(l8_col))); B=_to_days(s.get(s30_col)); C=np.sort(_to_days(s.get(l7_col)))
        l7_near=sum(np.searchsorted(A,x+tol_days,"right")>np.searchsorted(A,x-tol_days,"left") for x in C) if A.size and C.size else 0
        base=np.intersect1d(A,B); l8_eq=int(base.size)
        l8eq_with_l7=sum(np.searchsorted(C,d+tol_days,"right")>np.searchsorted(C,d-tol_days,"left") for d in base) if base.size and C.size else 0
        return {"FID":s.get("FID",s.get("_row_id")),"l7_within_tol_of_l8":int(l7_near),"l8_equal_s30":l8_eq,"l8_equal_s30_with_l7_tol":int(l8eq_with_l7)}
    idx=list(gdf.index); cols=[c for c in [l7_col,l8_col,s30_col,"FID"] if c in gdf.columns]; recs=gdf[cols].to_dict("records")
    for i,r in enumerate(recs): r["_row_id"]=idx[i] if "FID" not in r else r.get("FID",idx[i])
    stats=pd.DataFrame(Parallel(n_jobs=njobs)(delayed(_counts_row)(r) for r in recs))
    if stats_csv: os.makedirs(os.path.dirname(stats_csv) or ".",exist_ok=True); stats.to_csv(stats_csv,index=False)
    out=gdf.copy(); before=len(out)
    vals=Parallel(n_jobs=njobs)(delayed(lambda r:_row(r.get(l7_col),r.get(l8_col),r.get(s30_col)))(r) for r in recs)
    out[out_col]=vals
    out["l8_equal_s30"]=stats["l8_equal_s30"].to_numpy()
    out["l7_within_tol_of_l8"]=stats["l7_within_tol_of_l8"].to_numpy()
    # keep all rows; only drop source date cols when requested
    out=out.reset_index(drop=True)
    if remove_empty:
        out=out.drop(columns=[l7_col,l8_col,s30_col],errors="ignore")
        out=out[out[out_col].map(bool)].reset_index(drop=True)
    total_eq=int(stats["l8_equal_s30"].sum()); total_eq_with_l7=int(stats["l8_equal_s30_with_l7_tol"].sum())
    print(f"rows: {before} → {len(out)}"); print(f"L8==S30 exact-date matches: {total_eq}"); print(f"…of these, with L7 within ±{tol_days} day(s): {total_eq_with_l7}")
    return out
def _ensure_list(x):
    if isinstance(x,(list,tuple,np.ndarray)): return list(x)
    if pd.isna(x): return []
    return re.findall(r'\d{4}-\d{2}-\d{2}',str(x))
def make_shuffle_gdf(src,seed=None):
    gdf=src if isinstance(src,gpd.GeoDataFrame) else gpd.read_file(src)
    gdf=gdf.copy(); rng=random.Random(seed)
    gdf['good_d']=gdf['good_d'].apply(_ensure_list).apply(lambda L: rng.shuffle(L) or L)
    return gdf
# --- QA+SR filters ---
def check_radsat_bits_clear(df,col='QA_RADSAT',bits=range(7)):
    if col not in df: return pd.Series(False,index=df.index)
    s=pd.to_numeric(df[col],errors='coerce').fillna(0).astype(int)
    bitmask=sum(1<<b for b in bits); return (s & bitmask)==0
def check_qapixel_whitelist(df,col='QA_PIXEL',allowed=ALLOWED_QA_PIXEL):
    if col not in df: return pd.Series(False,index=df.index)
    s=pd.to_numeric(df[col],errors='coerce').astype('Int64'); return s.isin(allowed)
def drop_cols_with_ST(df):
    return df.drop(columns=[c for c in df.columns if 'ST' in str(c).upper()],errors='ignore')
def scale_and_filter_Bcols(df):
    out=df.copy(); bcols=[c for c in out.columns if re.search(r'B\d+',str(c))]
    if not bcols: return out.reset_index(drop=True)
    for c in bcols: out[c]=pd.to_numeric(out[c],errors='coerce')*0.0000275-0.2
    m=((out[bcols]>=0)&(out[bcols]<=1)).all(axis=1)
    return out.loc[m].reset_index(drop=True)
def apply_Landsat_filters(samples):
    m1=check_radsat_bits_clear(samples); m2=check_qapixel_whitelist(samples)
    samples=samples.loc[m1 & m2].reset_index(drop=True)
    samples=drop_cols_with_ST(samples)
    return scale_and_filter_Bcols(samples)
def filter_by_percentage_difference(L7vals,L8vals):
    Blue_diff=abs((L7vals['SR_B1']-L8vals['SR_B2'])/abs(L7vals['SR_B1']+L8vals['SR_B2']*.5))
    if Blue_diff< 1:
        return True,Blue_diff
    else:
        return False
    return False, Blue_diff
def apply_hls_filter(df,fmask_col='Fmask',bprefix='B'):
    if fmask_col not in df: return df.iloc[0:0].copy()
    s=pd.to_numeric(df[fmask_col],errors='coerce')
    x=s.fillna(-1).astype('int64').to_numpy()                 # NumPy for safe bit ops
    m=((x>>1)&1==0)&((x>>2)&1==0)&((x>>3)&1==0)&(((x>>6)&3)==1)  # no cloud/adj/shadow & low aerosol
    m=pd.Series(m,index=df.index)
    bcols=[c for c in df.columns if str(c).startswith(bprefix)]
    if bcols:
        bnum=df[bcols].apply(pd.to_numeric,errors='coerce')
        m=m&((bnum>=0)&(bnum<=1)).all(axis=1)                 # all B* in [0,1]
    return df.loc[m].reset_index(drop=True)
def _ic_for_date(col, point_ll, start, end):
    return (ee.ImageCollection(col)
            .filterDate(start, end)
            .filterBounds(point_ll))
def _sample_one_date(col, point_ll, date_str, bands,scale=30):
    """Return dict of sampled values or None."""
    d=pd.to_datetime(date_str); start=d.strftime('%Y-%m-%d'); end=(d+timedelta(days=1)).strftime('%Y-%m-%d')
    ic=_ic_for_date(col, point_ll, start, end)
    try:
        if ic.size().getInfo()==0: return None
        img=(ic.first()
             .select(bands))
        vals=img.reduceRegion(ee.Reducer.first(), geometry=point_ll, scale=scale,
                              bestEffort=True, maxPixels=1e8).getInfo()
        return vals if vals else None
    except Exception:
        return None
def _sample_date_range(col,point_ll,date_str,bands,date_range=1,scale=10):
    """Sample within ±date_range days of date_str; prefer closest date."""
    d=pd.to_datetime(date_str)
    start=(d-pd.Timedelta(days=date_range)).strftime('%Y-%m-%d')
    end=(d+pd.Timedelta(days=date_range+1)).strftime('%Y-%m-%d')  # end exclusive
    ic=_ic_for_date(col,point_ll,start,end).select(bands).sort('system:time_start')
    try:
        n=ic.size().getInfo()
        if n==0: return None
        ts=ic.aggregate_array('system:time_start').getInfo()
        target_ms=int(pd.Timestamp(d).value/10**6)
        order=sorted(range(n),key=lambda i:abs(ts[i]-target_ms))  # closest-first
        L=ic.toList(n)
        for i in order:
            img=ee.Image(L.get(i))
            vals=img.reduceRegion(ee.Reducer.first(),geometry=point_ll,scale=scale,bestEffort=True,maxPixels=1e8).getInfo()
            if vals and not pd.isna(vals.get('label')): return vals
        return None
    except Exception:
        return None
# ---------------- validators ----------------
def validate_L7_date(point_ll, date_str,Landsat_bands):
    vals=_sample_one_date(L7COL, point_ll, date_str,Landsat_bands)
    if not vals: return False, None
    df=apply_Landsat_filters(pd.DataFrame([vals]))
    return (not df.empty, (None if df.empty else df.iloc[0].to_dict()))
def validate_L8_date(point_ll, date_str,Landsat_bands,day_offsets=(-1,1)):
    d0=pd.to_datetime(date_str)#try L8 with both 1 day offsets
    for off in day_offsets:
        d_try=d0+pd.Timedelta(days=off)
        vals=_sample_one_date(L8COL, point_ll, d_try.strftime('%Y-%m-%d'), Landsat_bands)
        if not vals: continue
        df=apply_Landsat_filters(pd.DataFrame([vals]))
        if df.empty: continue
        return True, df.iloc[0].to_dict(), d_try.strftime('%Y-%m-%d')
    return False, None, None
def validate_HLS_date(HLS_COL,point_ll, date_str,HLS_bands):
    vals=_sample_one_date(HLS_COL, point_ll, date_str,HLS_bands)
    if not vals: return False, None
    df=apply_hls_filter(pd.DataFrame([vals]))
    return (not df.empty, (None if df.empty else df.iloc[0].to_dict()))
def validate_HLS_date(HLS_COL,point_ll, date_str,HLS_bands):
    vals=_sample_one_date(HLS_COL, point_ll, date_str,HLS_bands)
    if not vals: return False, None
    df=apply_hls_filter(pd.DataFrame([vals]))
    return (not df.empty, (None if df.empty else df.iloc[0].to_dict()))
def validate_DW_date(DW_COL,point_ll,date_str,DW_band,scale=10,date_range=10):
    vals=_sample_date_range(DW_COL,point_ll,date_str,DW_band,date_range=date_range,scale=scale)
    return (False,None) if (not vals or pd.isna(vals.get('label'))) else (True,vals['label'])
# ---------------- small utility ----------------
def _prefix_pixel_cols(d, prefix):
    """Prefix only bands B* and QA_*; keep other keys unchanged if present."""
    out={}
    for k,v in (d or {}).items():
        if re.search(r'B\d+', str(k)):
            out[f'{prefix}{k}']=v
        else:
            out[k]=v
    return out
# ---------------- main driver ----------------
@contextmanager
def _tqdm_joblib(t):
    # bridge joblib → tqdm; always restore original callback
    class _CB(joblib.parallel.BatchCompletionCallBack):
        def __call__(self,*a,**k): t.update(n=self.batch_size); return super().__call__(*a,**k)
    old=joblib.parallel.BatchCompletionCallBack; joblib.parallel.BatchCompletionCallBack=_CB
    try: yield t
    finally: joblib.parallel.BatchCompletionCallBack=old; t.close()
def sample_dates(shuffled_dates_gdf,**kw):
    # EE init moved out of here (done once in parallel_sample_dates)
    t0=time.perf_counter()
    cnt={'assessed':0,'L7_false':0,'L8_false':0,'blue_false':0,'S30_false':0,'L30_false':0,'DW_false':0}
    out=[]; G=shuffled_dates_gdf  # input is EPSG:4326 already; no to_crs
    def _safe_unpack(ret,n):
        if isinstance(ret,tuple): return ret+(None,)*(n-len(ret)) if len(ret)<n else ret[:n]
        if isinstance(ret,bool): return (ret,)+(None,)*(n-1)
        return (False,)+(None,)*(n-1)
    def _iter_dates(v):
        if v is None or (isinstance(v,float) and pd.isna(v)): return []
        return list(v) if isinstance(v,(list,tuple,np.ndarray,pd.Series)) else [v]
    for i,r in G.iterrows():
        fid=r.get('FID',i); geom=r.geometry
        if geom is None or geom.is_empty: continue
        pt=ee.Geometry.Point(geom.x,geom.y)
        for d in _iter_dates(r.get('good_d',[])):
            if d is None or (isinstance(d,float) and pd.isna(d)): continue
            cnt['assessed']+=1
            L7ok,L7vals=_safe_unpack(validate_L7_date(pt,d,Landsat_bands),2)
            if not L7ok: cnt['L7_false']+=1; continue
            L8ok,L8vals,L8date=_safe_unpack(validate_L8_date(pt,d,Landsat_bands),3)
            if not L8ok or L8date is None: cnt['L8_false']+=1; continue
            bluediffok,bluediff=_safe_unpack(filter_by_percentage_difference(L7vals,L8vals),2)
            if not bluediffok: cnt['blue_false']+=1; continue
            S30ok,S30vals=_safe_unpack(validate_HLS_date(S30COL,pt,L8date,S30_Bands),2)
            if not S30ok: cnt['S30_false']+=1; continue
            L30ok,L30vals=_safe_unpack(validate_HLS_date(L30COL,pt,L8date,L30_Bands),2)
            if not L30ok: cnt['L30_false']+=1; continue
            DWok,DWval=_safe_unpack(validate_DW_date(DWCOL,pt,L8date,DW_band,scale=10),2)
            if not DWok: cnt['DW_false']+=1; continue
            rec={'FID':fid,'L7_Date':pd.to_datetime(d).strftime('%Y-%m-%d'),
                 'L8_date':pd.to_datetime(L8date).strftime('%Y-%m-%d'),
                 'DW_label':DWval}
            rec.update(_prefix_pixel_cols(L7vals,'L7_')); rec.update(_prefix_pixel_cols(L8vals,'L8_'))
            rec.update(_prefix_pixel_cols(S30vals,'S30_')); rec.update(_prefix_pixel_cols(L30vals,'L30_'))
            out.append(rec); break
    df=pd.DataFrame(out).reset_index(drop=True)
    return df,{**cnt,'rows':len(G),'ok_records':len(df),'sec':time.perf_counter()-t0}
def parallel_sample_dates(gdf,out_shp,log_path,n_jobs=-1,chunk_size=5,**kw):
    # EE: initialize once (threads share the same process)
    try: ee.Initialize(project=ee_project)
    except Exception: 
        try: ee.Initialize()
        except Exception as e: raise e
    G=gpd.GeoDataFrame(gdf,geometry='geometry'); crs=G.crs; n=len(G)
    if n==0:
        empty=gpd.GeoDataFrame(columns=['FID','geometry'],crs=crs)
        try: empty.to_file(out_shp)
        except Exception: pass
        os.makedirs(os.path.dirname(log_path),exist_ok=True)
        with open(log_path,'w') as f: f.write("No rows to process.\n")
        return empty
    idx=[np.arange(n)[i:i+chunk_size] for i in range(0,n,chunk_size)] if chunk_size else np.array_split(np.arange(n),max(1,(os.cpu_count() if n_jobs in(-1,None) else int(n_jobs))))
    parts=[G.iloc[i] for i in idx if len(i)]
    t0=time.perf_counter()
    # Use threads: EE calls are IO-bound
    with _tqdm_joblib(tqdm(total=len(parts),desc="Sampling dates",unit="chunk")):
        res=Parallel(n_jobs=n_jobs,prefer="threads")(delayed(sample_dates)(p,**kw) for p in parts)
    dfs=[r[0] for r in res if r and r[0] is not None and len(r[0])]
    stats=[r[1] for r in res if r and r[1] is not None]
    if not dfs:
        out=gpd.GeoDataFrame(columns=['FID','geometry'],crs=crs)
    else:
        df=pd.concat(dfs,ignore_index=True)
        if 'geometry' not in df.columns and 'FID' in G.columns: df=df.merge(G[['FID','geometry']],on='FID',how='left')
        if 'L7_Date' in df.columns: df['L7_Date']=pd.to_datetime(df['L7_Date'],errors='coerce').dt.strftime('%Y-%m-%d')
        if 'L8_date' in df.columns: df['L8_date']=pd.to_datetime(df['L8_date'],errors='coerce').dt.strftime('%Y-%m-%d')
        out=gpd.GeoDataFrame(df,geometry='geometry',crs=crs)
    os.makedirs(os.path.dirname(out_shp),exist_ok=True)
    try: out.to_file(out_shp)
    except Exception as e: print(f"Write failed: {e}")
    agg={}
    for d in stats:
        for k,v in d.items(): agg[k]=agg.get(k,0)+v
    agg['total_rows']=n; agg['chunks']=len(parts); agg['total_sec']=time.perf_counter()-t0
    lines=[
        f"Total rows: {agg.get('total_rows',0)}  Chunks: {agg.get('chunks',0)}",
        f"Dates assessed: {agg.get('assessed',0)}  Accepted: {agg.get('ok_records',0)}",
        f"Continues — L7:false {agg.get('L7_false',0)}, L8:false {agg.get('L8_false',0)}, blue:false {agg.get('blue_false',0)},",
        f"             S30:false {agg.get('S30_false',0)}, L30:false {agg.get('L30_false',0)}, DW:false {agg.get('DW_false',0)}",
        f"Time — chunks sum (s): {round(agg.get('sec',0),3)}  wall (s): {round(agg.get('total_sec',0),3)}"
    ]
    os.makedirs(os.path.dirname(log_path),exist_ok=True)
    with open(log_path,'w') as f: f.write("\n".join(lines)+"\n")
    return out
def run_shp_folders(maindir, order="desc"):
    root=Path(maindir); fols=sorted([p for p in root.iterdir() if p.is_dir()], key=lambda p:p.name, reverse=order.lower().startswith("desc")); res=[]
    for f in fols:
        ee.Authenticate()
        shps=sorted(f.glob("*.shp")); 
        if not shps: print(f"[skip] {f.name}: no .shp"); continue
        shp=shps[0]; fn=shp.stem; out=str(f)
        print(f"processing {shp}")
        pg=list_tile_intersections(str(shp), WRS_shp,Sen2_shp,
                                   path_col="PATH",row_col="ROW",mgrs_col="Name",keep_point_cols=True,n_jobs=-1)
        pwd=attach_point_dates(pg,L7_dates,L8_dates,S30_dates)
        good=attach_good_dates_exact(pwd,stats_csv=str(Path(out)/"date_statistics.csv"))
        shuf=make_shuffle_gdf(good)
        out_shp=str(Path(out)/f"P{fn}.shp"); log=str(Path(out)/f"P{fn}.txt")
        print('Starting parallel processing')
        gdf=parallel_sample_dates(shuf,out_shp,log)
        res.append({"folder":f.name,"in_shp":str(shp),"out_shp":out_shp,"log":log,"gdf":gdf}); print(f"[ok] {f.name}: {shp.name} → {Path(out_shp).name}")
        print(f'completed {out_shp}')
    return res

The next cell runs the script. Put the shapefile in a folder, and set the folder path as the input parameter for run_shp_folders. If you want to process multiple shapefiles, ensure that they are all in seperate subdirs within maindir.

In [ ]:
run_shp_folders(r"E:\GIS\Landcover_sampling\Initial Points\1mill", order="asc")

processing E:\GIS\Landcover_sampling\Initial Points\2mill_init\10\inital2mil_(10).shp
rows: 100000 → 95797
L8==S30 exact-date matches: 4119181
…of these, with L7 within ±1 day(s): 2989817
Starting parallel processing


Sampling dates:   0%|          | 0/19160 [00:00<?, ?chunk/s]

Rename bands to colours. This is essential to do on the output shapefile in order to ensure that the everything is aligned for comparisons. The input for this funciton is the path of the shapefile you want to append the values.

In [ ]:
def remap_bands(gdp_file,out_path):
    S30={"B1":"CA","B2":"B","B3":"G","B4":"R","B8A":"NIR","B11":"SWIR1","B12":"SWIR2"}
    L8L30={"B1":"CA","B2":"B","B3":"G","B4":"R","B5":"NIR","B6":"SWIR1","B7":"SWIR2"}
    L7={"B1":"B","B2":"G","B3":"R","B4":"NIR","B5":"SWIR1","B7":"SWIR2"}
    gdf = gpd.read_file(gdp_file)
    gcol=getattr(gdf.geometry,'name','geometry')
    keep=[c for c in ("DW_label","L7_Date","L8_date",gcol) if c in gdf.columns]
    ren={}
    for c in gdf.columns:
        if c in keep: continue
        m=re.match(r'^(S30|L8|L30|L7)(?:_SR)?_(B\d{1,2}|B8A)$', c)
        if not m: continue
        s, b = m.group(1), m.group(2)
        mp = S30 if s=="S30" else (L8L30 if s in ("L8","L30") else L7)
        if b in mp: ren[c]=f"{s}_{mp[b]}"
    cols=keep+[c for c in gdf.columns if c in ren]
    out_gdf = gpd.GeoDataFrame(gdf[cols].rename(columns=ren), geometry=gcol, crs=getattr(gdf, 'crs', None))
    out_gdf.to_file(out_path, driver="ESRI Shapefile", index=False)
    return out_gdf

In [ ]:
remap_bands(r"E:\GIS\Landcover_sampling\Initial Points\1mill\1mill.shp",
            r"E:\GIS\Landcover_sampling\Initial Points\1mill\remapped1mill.shp")

If you want to add the "SR_ATMOS_OPACITY" and "SR_QA_AEROSOL" bands from the L7 and L8 sensors respectively, put your shapefile or shapefiles in a folder (ROOT parameter) and run this code.

In [ ]:
ROOT=r"E:\GIS\Landcover_sampling\Intermediate\To_add_atmos"
NJOBS=-1
def _ee_init():
    try: ee.Image(1).getInfo()
    except:
        try: ee.Initialize(project=ee_project)
        except: ee.Authenticate();ee.Initialize(project=ee_project)
def _img_for_date(coll,ds,area,window_days=2):
    t=pd.to_datetime(ds)
    for w in range(window_days+1):
        ic=(ee.ImageCollection(coll)
             .filterDate((t-pd.Timedelta(days=w)).strftime("%Y-%m-%d"),(t+pd.Timedelta(days=w+1)).strftime("%Y-%m-%d"))
             .filterBounds(area).sort("CLOUD_COVER"))
        try:
            if ic.size().getInfo()>0: return ic.first()
        except: _ee_init(); 
        try:
            if ic.size().getInfo()>0: return ic.first()
        except: pass
    return None
def _sample_group(coll,band,xy_idx,date,scale):
    _ee_init();area=ee.Geometry.MultiPoint([[x,y] for (x,y,_) in xy_idx]).bounds().buffer(1000);im=_img_for_date(coll,date,area)
    if not im: return []
    feats=[ee.Feature(ee.Geometry.Point(x,y),{"i":int(i)}) for (x,y,i) in xy_idx]
    try:
        fs=im.select(band).sampleRegions(ee.FeatureCollection(feats),scale=scale,geometries=False).getInfo()["features"]
        return [(f["properties"]["i"],f["properties"].get(band,np.nan)) for f in fs]
    except: return []
def sample_band_by_date(gdf,date_col,coll,band,scale=30,nj=NJOBS,desc=""):
    ds=pd.to_datetime(gdf[date_col]).dt.strftime("%Y-%m-%d");xy=np.c_[gdf.geometry.x.values,gdf.geometry.y.values,np.arange(len(gdf))]
    groups={};[groups.setdefault(d,[]).append(tuple(xy[i])) for i,d in enumerate(ds)]
    dates=list(groups.items());pairs=sum(Parallel(n_jobs=nj,backend="loky")(delayed(_sample_group)(coll,band,groups[d],d,scale) for d,_ in tqdm(dates,desc=desc,total=len(dates))),[])
    out=np.full(len(gdf),np.nan,dtype="float32");[out.__setitem__(i,(np.nan if v is None else v)) for i,v in pairs];return out
for shp in sorted(Path(ROOT).glob("*.shp")):
    if shp.name.startswith("A_"): continue
    gdf=gpd.read_file(shp); 
    if gdf.crs is None or gdf.crs.to_epsg()!=4326: gdf=gdf.to_crs(4326)
    cols={c.lower():c for c in gdf.columns};d7=cols.get("l7_date");d8=cols.get("l8_date")
    if not d7 or not d8: raise KeyError("Expected columns: L7_Date and L8_date")
    print(f"\n {shp.name}  L7 SR_ATMOS_OPACITY");gdf["SR_ATMOS_OPACITY"]=sample_band_by_date(gdf,d7,L7COL,"SR_ATMOS_OPACITY",desc="L7 date groups")
    print(f" {shp.name}  L8 SR_QA_AEROSOL");gdf["SR_QA_AEROSOL"]=sample_band_by_date(gdf,d8,L8COL,"SR_QA_AEROSOL",desc="L8 date groups")
    out=shp.with_name("A_"+shp.name);gdf.to_file(out,driver="ESRI Shapefile");print("✓",out)